# Notebook 2 — DINOv3 Patch Similarity (Meta, self-distillation, satellite-pretrained)

### *Hands-On Session: Earth Embeddings for EO — Retrieval, Discovery, and Change-Oriented Search*

**Instructors:** Fuxun Yu, Rishi Madhok (TerraByte AI)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iceysteel/ESA-NASA-Workshop-2026/blob/main/Day%202/Track%202/Earth-Embeddings-EO/notebook_2_dinov3_patch_similarity.ipynb)

---

## What we're doing

In **Notebook 1** we used [Prithvi-EO-2.0](https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-2.0-300M) — a **Masked Autoencoder** trained on 6-band, multi-temporal HLS imagery — to get per-patch embeddings and run a similarity heatmap.

Now we'll do the **same task on the same Mexico HLS scene** with a *very* different recipe:

**DINOv3** (Meta, 2025) — **self-distillation without labels**, trained on RGB imagery. Specifically we use the **SAT-493M** variant: DINOv3 ViT-L/16 *pretrained on 493M satellite images at ~0.6 m resolution* ([model card](https://huggingface.co/timm/vit_large_patch16_dinov3.sat493m)). 

Two big differences from Prithvi:

| | Prithvi-EO-2.0 (NB 1) | DINOv3 SAT-493M (this NB) |
|---|---|---|
| Objective | MAE — reconstruct masked patches | DINO — student/teacher self-distillation |
| Modality | 6-band multispectral + temporal | RGB only |
| Pretraining data | Global HLS (Landsat + Sentinel-2) | 493M Maxar high-res satellite images |
| Patch size | 16 px on 30 m HLS ≈ 480 m | 16 px on whatever resolution you feed it |

By running both notebooks on the same scene with the same query patches you can see, with your eyes, what each pretraining recipe considers 'similar.'

*Note: Meta's `facebook/dinov3-*` repos are gated. We use **timm's ungated mirror** of the same weights — no HF token, no access request needed.*


### Why DINOv3-SAT here, and where it sits among EO foundation models

The session brief names *AlphaEarth, Prithvi, TerraMind, and RS-CLIP* as representative EO foundation models. We picked these four to demo concretely:

| Model | Family | Notebook | Public weights? |
|---|---|---|---|
| **Prithvi-EO-2.0** (IBM-NASA, MAE) | masked self-supervised | NB1 | ✅ HF `ibm-nasa-geospatial/Prithvi-EO-2.0-300M` |
| **DINOv3 SAT-493M** (Meta, self-distillation) | self-supervised / contrastive | **this notebook** | ✅ HF mirror `timm/vit_large_patch16_dinov3.sat493m` |
| **Git-RSCLIP** (CLIP variant on Git10M) | multimodal alignment | NB3 / NB4 / NB5 | ✅ HF `lcybuaa/Git-RSCLIP-base` |
| **AlphaEarth** (Google DeepMind) | multi-sensor distillation | – | ❌ weights not released; only the precomputed 64-d *Satellite Embedding* asset on Earth Engine (`GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL`) |
| **TerraMind** (IBM / ESA) | any-to-any multimodal generative | – | ✅ HF org `ibm-esa-geospatial` (Tiny/Small/Base/Large, ungated); workshop-time we point to it rather than load it because the canonical loader is the `terratorch` package which adds significant install footprint |

DINOv3-SAT stands in for the **self-distillation** family on the same Mexico scene as Prithvi (NB1), giving you a real side-by-side comparison of how training objective shapes which textures and structures cluster together in the embedding space.


## Runtime

**Recommended:** `Runtime → Change runtime type → T4 GPU` (free) — ~15 s end-to-end. 
**Fallback:** CPU — ~30 s. Single forward pass on one image is cheap even for ViT-L.


In [ ]:
!pip install -q timm tifffile


In [ ]:
import time
import numpy as np
import torch
import torch.nn.functional as F
import timm
import tifffile
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from huggingface_hub import hf_hub_download

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)


## 1. Load DINOv3 ViT-L/16 SAT-493M

We load directly via `timm.create_model('hf_hub:timm/vit_large_patch16_dinov3.sat493m', ...)`. `timm` resolves the model's *own* preprocessing config (input size, mean/std) — these are **satellite-specific** statistics, not ImageNet's.


In [ ]:
MODEL_ID = 'hf_hub:timm/vit_large_patch16_dinov3.sat493m'

t0 = time.time()
model = timm.create_model(MODEL_ID, pretrained=True).eval().to(device)
print(f'Loaded in {time.time()-t0:.1f}s.')

data_cfg = timm.data.resolve_model_data_config(model)
transform = timm.data.create_transform(**data_cfg, is_training=False)

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'DINOv3 ViT-L/16 SAT-493M — {n_params:.1f}M params, embed_dim={model.num_features}')
print('Input size :', data_cfg['input_size'])
print('Mean / Std :', data_cfg['mean'], '/', data_cfg['std'])
print('Reg tokens :', getattr(model, "num_reg_tokens", 0) or getattr(model, "reg_token", None) is not None)


## 2. Use the SAME HLS scene as Notebook 1, rendered as RGB

DINOv3 takes RGB. We:

1. Download one of Prithvi's bundled Mexico HLS tiles (band order `[B02, B03, B04, B05, B06, B07]`).
2. Compose RGB from `(B04, B03, B02)` and contrast-stretch to uint8.
3. Center-crop to 224×224 — same spatial extent as Notebook 1, but DINOv3 will resize to its native 256×256.


In [ ]:
tif_path = hf_hub_download(
    'ibm-nasa-geospatial/Prithvi-EO-2.0-300M',
    'examples/Mexico_HLS.S30.T13REM.2018201T172901.v2.0_cropped.tif',  # Apr 2018
)
raw = tifffile.imread(tif_path)        # (H, W, 6)
print('Raw chip shape:', raw.shape, 'dtype:', raw.dtype)

S = 224
h0, w0 = (raw.shape[0] - S) // 2, (raw.shape[1] - S) // 2
chip = raw[h0:h0+S, w0:w0+S]            # (224, 224, 6)

def to_rgb(chip_hwc, p_low=2, p_high=98):
    rgb = chip_hwc[..., [2, 1, 0]].astype(np.float32)   # R=B04, G=B03, B=B02
    lo, hi = np.percentile(rgb, [p_low, p_high])
    return np.clip((rgb - lo) / max(hi - lo, 1e-6), 0, 1)

rgb01 = to_rgb(chip)
rgb_u8 = (rgb01 * 255).astype(np.uint8)

plt.figure(figsize=(5, 5))
plt.imshow(rgb01); plt.axis('off')
plt.title('Mexico HLS, Apr 2018 — RGB (B04, B03, B02), 224x224')
plt.show()


## 3. Extract per-patch embeddings

`timm`'s ViT models expose `model.forward_features(x)`, which returns **all output tokens** of the last block: `[CLS] [reg_1 … reg_R] [patch_1 … patch_{P*P}]`. We slice off the prefix tokens to get only the spatial patch tokens.

At DINOv3's native 256×256 input with patch size 16, that's a **16×16 = 256-patch** spatial grid of 1024-d features.


In [ ]:
from PIL import Image
img_pil = Image.fromarray(rgb_u8)
x = transform(img_pil).unsqueeze(0).to(device)    # (1, 3, 256, 256)
print('Model input tensor:', tuple(x.shape))

with torch.inference_mode():
    t0 = time.time()
    tokens = model.forward_features(x)            # (1, 1 + R + P*P, D)
    print(f'Forward pass: {time.time()-t0:.2f}s on {device}')

tokens = tokens[0]                                # (T, D)
n_prefix = tokens.shape[0] - 16 * 16              # CLS + register tokens, computed from grid size
print(f'Total tokens: {tokens.shape[0]} (= {n_prefix} CLS/register + 256 patch tokens)')

patch_tokens = tokens[n_prefix:]                  # (256, D)
spatial = patch_tokens.reshape(16, 16, -1).cpu().numpy()
print('Spatial feature map:', spatial.shape)


## 4. Patch-similarity heatmap

Same recipe as Notebook 1: pick a query patch, cosine-sim against every other patch, upsample, overlay.

**Note** the grid is now 16×16 (DINOv3 at 256×256), not 14×14 (Prithvi at 224×224). The image area is the same, so each patch covers a slightly smaller piece of ground.


In [ ]:
QUERY_RC = (7, 7)         # row, col in the 16x16 patch grid
G = 16                    # grid side
PS = data_cfg['input_size'][1] // G   # 256 / 16 = 16 px per patch

feat = torch.from_numpy(spatial).reshape(-1, spatial.shape[-1])
feat = F.normalize(feat, dim=-1)
q = feat[QUERY_RC[0] * G + QUERY_RC[1]]
sim = (feat @ q).reshape(G, G).numpy()
print(f'Similarity range: {sim.min():.3f} .. {sim.max():.3f}')

# We display the heatmap over the *original 224x224 RGB*. Resize sim to 224x224.
sim_up = F.interpolate(torch.from_numpy(sim)[None, None].float(),
                       size=(S, S), mode='bilinear', align_corners=False).squeeze().numpy()
# Map the patch coord from the 256x256 grid back to 224x224 RGB pixels for the rectangle.
scale = S / data_cfg['input_size'][1]
qy, qx = int(QUERY_RC[0] * PS * scale), int(QUERY_RC[1] * PS * scale)
rect_side = int(PS * scale)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(rgb01); axes[0].set_title('RGB')
axes[0].add_patch(Rectangle((qx, qy), rect_side, rect_side, ec='cyan', fc='none', lw=2.5))
axes[1].imshow(sim, cmap='magma'); axes[1].set_title(f'Raw 16x16 cosine sim (query={QUERY_RC})')
axes[1].add_patch(Rectangle((QUERY_RC[1] - 0.5, QUERY_RC[0] - 0.5), 1, 1, ec='cyan', fc='none', lw=2))
axes[2].imshow(rgb01); axes[2].imshow(sim_up, cmap='magma', alpha=0.55)
axes[2].add_patch(Rectangle((qx, qy), rect_side, rect_side, ec='cyan', fc='none', lw=2.5))
axes[2].set_title('RGB + similarity overlay')
for ax in axes: ax.axis('off')
plt.suptitle('DINOv3 SAT-493M patch similarity (cosine, last encoder layer)', y=1.02)
plt.tight_layout(); plt.show()


### Multiple query patches — same locations as Notebook 1 (scaled for 16x16 grid)

In [ ]:
# In NB 1 we queried (2,2), (6,6), (10,10), (3,11) on a 14x14 grid.
# queries = [(1, 1), (6, 6), (6, 8), (3, 11)]
# Map roughly to the equivalent spatial positions on the 16x16 grid here.
queries = [(2, 2), (7, 7), (7, 9), (3, 13)]
colors = ['cyan', 'lime', 'orange', 'magenta']

fig, axes = plt.subplots(1, len(queries) + 1, figsize=(4.2 * (len(queries) + 1), 4.2))
axes[0].imshow(rgb01); axes[0].set_title('RGB (Apr 2018)')
for (qr, qc), color in zip(queries, colors):
    qy2 = int(qr * PS * scale); qx2 = int(qc * PS * scale)
    axes[0].add_patch(Rectangle((qx2, qy2), rect_side, rect_side, ec=color, fc='none', lw=2))
axes[0].axis('off')

for ax, (qr, qc), color in zip(axes[1:], queries, colors):
    qvec = feat[qr * G + qc]
    s = (feat @ qvec).reshape(G, G).numpy()
    s_up = F.interpolate(torch.from_numpy(s)[None, None].float(),
                          size=(S, S), mode='bilinear', align_corners=False).squeeze().numpy()
    qy2 = int(qr * PS * scale); qx2 = int(qc * PS * scale)
    # ax.imshow(rgb01); 
    ax.imshow(s_up, cmap='magma', alpha=0.55)
    ax.add_patch(Rectangle((qx2, qy2), rect_side, rect_side, ec=color, fc='none', lw=2.5))
    ax.set_title(f'query=({qr},{qc})'); ax.axis('off')

plt.suptitle('DINOv3 SAT-493M — patch similarity from 4 different query locations', y=1.02)
plt.tight_layout(); plt.show()


## 5. Side-by-side intuition — Prithvi vs DINOv3

After running both notebooks on the same scene with the same approximate query points, you'll typically observe:

- **Prithvi** (multispectral + temporal MAE): groups patches by *reflectance and temporal trajectory*. Vegetation that greens-up at the same time of year clusters together even when the RGB looks different.
- **DINOv3 SAT-493M** (RGB self-distillation): groups patches by *visual texture / structure*. Tends to be sharper on built structures, roads, and field boundaries, but blind to anything outside the RGB envelope (e.g., NIR vegetation health, SWIR moisture).

Neither is 'correct' — they are answering different similarity questions. Choosing between them (or combining them) is one of the core design decisions in operational EO embedding systems.

## 6. Takeaways

- **Self-distillation (DINO) is a viable alternative to MAE** for producing dense, per-patch features without labels.
- **Pretraining data dominates downstream behavior.** Same backbone, same objective, but DINOv3 SAT-493M (satellite-pretrained) produces noticeably different features than DINOv3 LVD-1689M (web-image-pretrained) — for EO work, prefer the satellite variant when available.
- **Register tokens** are a DINOv3 architectural detail you'll see in any modern ViT — extra learnable tokens that absorb 'noise' attention and improve dense feature quality. Always slice them off when extracting patch features.
- **Modalities matter.** Going from 6-band HLS (Prithvi) to RGB (DINOv3) throws away information that EO scientists often care most about (NIR/SWIR vegetation, moisture, burn signatures). A workflow may want both, indexed in parallel, with the right index queried for the right question.
- **Where this is heading in the session:**
  - Notebook 3 will index embeddings at scale with an **ANN** structure (FAISS).
  - Notebook 4 will swap raw patches for **text queries** (RS-CLIP) — *describe* what you're looking for instead of clicking on it.
